# Plant Species and Disease Detection
## GOAL 3: Species AND Disease Detection

In [1]:
# FOR RUNNING LOCALLY
! pip install matplotlib opencv-python numpy seaborn pandas tqdm scikit-learn kagglehub scikit-image


### STEP 0: Load dataset, examine data, and preprocess

In [2]:
import matplotlib.pyplot as plt
import cv2 as cv
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import time
import json

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [3]:
#LOCAL VERSION
! pip install python-dotenv
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get('KAGGLE_API_TOKEN'):
    print("Error: Could not find 'KAGGLE_API_TOKEN'")

In [4]:
! pip install kagglehub
import kagglehub

path = kagglehub.dataset_download("riteshmaurya86/cleaned-plant-disease-image-dataset")

print("Dataset downloaded to:", path)

/media/mahi/Shared/AUC/Semesters/6.5- Summer 2026/Fundamentals of Computer Vision/Project/Plant-Disease-Detection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset downloaded to: /home/mahi/.cache/kagglehub/datasets/riteshmaurya86/cleaned-plant-disease-image-dataset/versions/1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/PlantDetection'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
# LOCAL VERSION
import os
OUTPUT_DIR = os.path.expanduser('outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [6]:
# we only need this to get the class folder names
DATA_DIR = os.path.join(path, 'data')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VALID_DIR = os.path.join(DATA_DIR, 'valid')
TEST_DIR = os.path.join(DATA_DIR, 'test')

classes = sorted(os.listdir(TRAIN_DIR))
valid_classes = sorted(os.listdir(VALID_DIR))
print(f"Number of classes: {len(classes)}")

Number of classes: 38


## STEP 1: Classical ML
#### First: Prep data
Features were already extractred and saved from part 1

In [7]:

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [9]:
FEATURES_PATH = os.path.join(OUTPUT_DIR, 'classical_ml_features.npz')
VALID_PATH = os.path.join(OUTPUT_DIR, 'classical_ml_features_valid.npz')
TEST_PATH = os.path.join(OUTPUT_DIR, 'classical_ml_features_test.npz')

train_data = np.load(FEATURES_PATH, allow_pickle=True)
X_train, meta_train = train_data['X'], train_data['meta']

valid_data = np.load(VALID_PATH, allow_pickle=True)
X_valid, meta_valid = valid_data['X'], valid_data['meta']

test_data = np.load(TEST_PATH, allow_pickle=True)
X_test, meta_test = test_data['X'], test_data['meta']

print(f"Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")

Train: (70294, 121), Valid: (17572, 121), Test: (33, 121)


In [12]:
#extract species from label
def get_species(class_name):
    #Species is the part before ___
    return class_name.split("___")[0]

SPECIES_LIST = sorted({get_species(c) for c in classes})
print(f"{len(SPECIES_LIST)} species: {SPECIES_LIST}")

SPECIES_KEYWORDS = {
    "Apple": "Apple", "Blueberry": "Blueberry", "Cherry_(including_sour)": "Cherry",
    "Corn_(maize)": "Corn", "Grape": "Grape", "Orange": "Orange", "Peach": "Peach",
    "Pepper,_bell": "Pepper", "Potato": "Potato", "Raspberry": "Raspberry",
    "Soybean": "Soybean", "Squash": "Squash", "Strawberry": "Strawberry", "Tomato": "Tomato",
}

#for the flat test folder
def get_species_from_filename(filename):
    for species, keyword in SPECIES_KEYWORDS.items():
        if keyword.lower() in filename.lower():
            return species
    return None

14 species: ['Apple', 'Blueberry', 'Cherry_(including_sour)', 'Corn_(maize)', 'Grape', 'Orange', 'Peach', 'Pepper,_bell', 'Potato', 'Raspberry', 'Soybean', 'Squash', 'Strawberry', 'Tomato']


In [10]:
# labels
y_train_full = np.array([m.split('/')[0] for m in meta_train])
y_valid_full = np.array([m.split('/')[0] for m in meta_valid])

print(pd.Series(y_train_full).value_counts())

Soybean___healthy                                     2021
Apple___Apple_scab                                    2016
Orange___Haunglongbing_(Citrus_greening)              2010
Apple___healthy                                       2008
Pepper,_bell___healthy                                1988
Apple___Black_rot                                     1987
Tomato___Tomato_Yellow_Leaf_Curl_Virus                1961
Potato___Early_blight                                 1939
Potato___Late_blight                                  1939
Tomato___healthy                                      1926
Grape___Esca_(Black_Measles)                          1920
Tomato___Early_blight                                 1920
Pepper,_bell___Bacterial_spot                         1913
Corn_(maize)___Northern_Leaf_Blight                   1908
Corn_(maize)___Common_rust_                           1907
Grape___Black_rot                                     1888
Tomato___Leaf_Mold                                    18

In [ ]:
# disease keyword per class
DISEASE_KEYWORDS = {
    "Apple___Apple_scab": "scab", "Apple___Black_rot": "rot", "Apple___Cedar_apple_rust": "rust",
    "Apple___healthy": "healthy",
    "Blueberry___healthy": "healthy",
    "Cherry_(including_sour)___Powdery_mildew": "mildew", "Cherry_(including_sour)___healthy": "healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "Cercospora", "Corn_(maize)___Common_rust_": "rust",
    "Corn_(maize)___Northern_Leaf_Blight": "Blight", "Corn_(maize)___healthy": "healthy",
    "Grape___Black_rot": "rot", "Grape___Esca_(Black_Measles)": "Esca",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)": "Isariopsis", "Grape___healthy": "healthy",
    "Orange___Haunglongbing_(Citrus_greening)": "greening",
    "Peach___Bacterial_spot": "Bacterial", "Peach___healthy": "healthy",
    "Pepper,_bell___Bacterial_spot": "Bacterial", "Pepper,_bell___healthy": "healthy",
    "Potato___Early_blight": "Early", "Potato___Late_blight": "Late", "Potato___healthy": "healthy",
    "Raspberry___healthy": "healthy",
    "Soybean___healthy": "healthy",
    "Squash___Powdery_mildew": "mildew",
    "Strawberry___Leaf_scorch": "scorch", "Strawberry___healthy": "healthy",
    "Tomato___Bacterial_spot": "Bacterial", "Tomato___Early_blight": "Early", "Tomato___Late_blight": "Late",
    "Tomato___Leaf_Mold": "Mold", "Tomato___Septoria_leaf_spot": "Septoria",
    "Tomato___Spider_mites Two-spotted_spider_mite": "Spider", "Tomato___Target_Spot": "Target",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "Curl", "Tomato___Tomato_mosaic_virus": "mosaic",
    "Tomato___healthy": "healthy",
}

def get_full_label_from_filename(filename):
    for class_name, disease_kw in DISEASE_KEYWORDS.items():
        species_kw = SPECIES_KEYWORDS[get_species(class_name)]
        if species_kw.lower() in filename.lower() and disease_kw.lower() in filename.lower():
            return class_name
    return None  # will need manual work

y_test_full = np.array([get_full_label_from_filename(m) for m in meta_test])

unmatched = [m for m, y in zip(meta_test, y_test_full) if y is None]
print(f"{len(unmatched)} unmatched filenames (need manual labeling): {unmatched}")

0 unmatched filenames (need manual labeling): []


#### Now scale and train

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingClassifier(random_state=42),
}

full_results = {}
for name, model in models.items():
    start = time.time()
    model.fit(X_train_scaled, y_train_full)
    preds = model.predict(X_valid_scaled)
    elapsed = time.time() - start
    acc = accuracy_score(y_valid_full, preds)
    f1 = f1_score(y_valid_full, preds, average='macro')
    full_results[name] = {'model': model, 'acc': acc, 'f1': f1, 'time': elapsed}
    print(f"{name}: acc={acc:.3f}, f1_macro={f1:.3f}, time={elapsed:.1f}s")

best_full_name = max(full_results, key=lambda n: full_results[n]['f1'])
best_full_model = full_results[best_full_name]['model']
print(f"\nBest model: {best_full_name}")

Logistic Regression: acc=0.850, f1_macro=0.849, time=300.6s
KNN: acc=0.855, f1_macro=0.854, time=8.3s
Random Forest: acc=0.942, f1_macro=0.941, time=125.3s
Hist Gradient Boosting: acc=0.951, f1_macro=0.951, time=177.9s

Best model: Hist Gradient Boosting


#### Test on the small test set and save report

In [19]:
preds = best_full_model.predict(X_test_scaled)
print(classification_report(y_test_full, preds))

report = classification_report(y_test_full, preds, output_dict=True)
with open(os.path.join(OUTPUT_DIR, 'classical_ml_full_summary.json'), 'w') as f:
    json.dump({
        "best_model": best_full_name,
        "comparison": {n: {"acc": r["acc"], "f1_macro": r["f1"]} for n, r in full_results.items()},
        "test_report": report,
    }, f, indent=2)

                                        precision    recall  f1-score   support

                    Apple___Apple_scab       1.00      1.00      1.00         3
              Apple___Cedar_apple_rust       1.00      1.00      1.00         4
           Corn_(maize)___Common_rust_       1.00      1.00      1.00         3
                Pepper,_bell___healthy       0.00      0.00      0.00         0
                 Potato___Early_blight       1.00      1.00      1.00         5
                      Potato___healthy       1.00      1.00      1.00         2
               Tomato___Bacterial_spot       0.00      0.00      0.00         0
                 Tomato___Early_blight       1.00      0.67      0.80         6
Tomato___Tomato_Yellow_Leaf_Curl_Virus       0.83      0.83      0.83         6
                      Tomato___healthy       1.00      1.00      1.00         4

                              accuracy                           0.91        33
                             macro avg

/media/mahi/Shared/AUC/Semesters/6.5- Summer 2026/Fundamentals of Computer Vision/Project/Plant-Disease-Detection/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/media/mahi/Shared/AUC/Semesters/6.5- Summer 2026/Fundamentals of Computer Vision/Project/Plant-Disease-Detection/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/media/mahi/Shared/AUC/Semesters/6.5- Summer 2026/Fundamentals of Computer Vision/Project/Plant-Disease-Detection/.venv/lib/python3.12/site-packages/sklearn

In [20]:
from sklearn.metrics import confusion_matrix
present_labels = sorted(set(y_test_full) | set(preds))
cm = confusion_matrix(y_test_full, preds, labels=present_labels)
pd.DataFrame(cm, index=present_labels, columns=present_labels)

,Apple___Apple_scab,Apple___Cedar_apple_rust,Corn_(maize)___Common_rust_,"Pepper,_bell___healthy",Potato___Early_blight,Potato___healthy,Tomato___Bacterial_spot,Tomato___Early_blight,Tomato___Tomato_Yellow_Leaf_Curl_Virus,Tomato___healthy
Apple___Apple_scab,3,0,0,0,0,0,0,0,0,0
Apple___Cedar_apple_rust,0,4,0,0,0,0,0,0,0,0
Corn_(maize)___Common_rust_,0,0,3,0,0,0,0,0,0,0
"Pepper,_bell___healthy",0,0,0,0,0,0,0,0,0,0
Potato___Early_blight,0,0,0,0,5,0,0,0,0,0
Potato___healthy,0,0,0,0,0,2,0,0,0,0
Tomato___Bacterial_spot,0,0,0,0,0,0,0,0,0,0
Tomato___Early_blight,0,0,0,0,0,0,1,4,1,0
Tomato___Tomato_Yellow_Leaf_Curl_Virus,0,0,0,1,0,0,0,0,5,0
Tomato___healthy,0,0,0,0,0,0,0,0,0,4


In [21]:
mismatched = [(fname, true, pred) for fname, true, pred in zip(meta_test, y_test_full, preds)
              if true == 'Tomato___Tomato_Yellow_Leaf_Curl_Virus' and pred == 'Pepper,_bell___healthy']
print(mismatched)

[(np.str_('TomatoYellowCurlVirus4.JPG'), np.str_('Tomato___Tomato_Yellow_Leaf_Curl_Virus'), np.str_('Pepper,_bell___healthy'))]


## STEP 2: DEEP LEARNING

#### First: Setup

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from collections import Counter

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cpu":
    print("WARNING: no GPU detected")

In [ ]:
class LeafFolderDataset(Dataset):
    def __init__(self, root_dir, class_list, label_fn, label_to_idx, transform=None):
        self.transform = transform
        self.samples = []

        for c in class_list:
            label = label_to_idx[label_fn(c)]
            class_dir = os.path.join(root_dir, c)
            for fname in os.listdir(class_dir):
                self.samples.append((os.path.join(class_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


class LeafFlatDataset(Dataset):
    def __init__(self, root_dir, label_fn, label_to_idx, transform=None):
        self.filenames = os.listdir(root_dir)
        self.root_dir = root_dir
        self.label_fn = label_fn
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        path = os.path.join(self.root_dir, fname)
        label = self.label_to_idx[self.label_fn(fname)]
        img = Image.open(path).convert('RGB')

        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
IMG_SIZE_DL = 224
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE_DL, IMG_SIZE_DL)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize(imagenet_mean, imagenet_std),
])

eval_transform = T.Compose([
    T.Resize((IMG_SIZE_DL, IMG_SIZE_DL)),
    T.ToTensor(),
    T.Normalize(imagenet_mean, imagenet_std),
])

#### Second: Prep data and models

In [ ]:
SPECIES_TO_IDX = {s: i for i, s in enumerate(SPECIES_LIST)}
IDX_TO_SPECIES = {i: s for s, i in SPECIES_TO_IDX.items()}

BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset_dl = LeafFolderDataset(TRAIN_DIR, classes, get_species, SPECIES_TO_IDX, transform=train_transform)
valid_dataset_dl = LeafFolderDataset(VALID_DIR, valid_classes, get_species, SPECIES_TO_IDX, transform=eval_transform)

train_loader = DataLoader(train_dataset_dl, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_dataset_dl, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train samples: {len(train_dataset_dl)}, Valid samples: {len(valid_dataset_dl)}")

In [ ]:
# note that theres an imbalance between species
label_counts = Counter([l for _, l in train_dataset_dl.samples])
total = sum(label_counts.values())
num_classes = len(SPECIES_LIST)

class_weights = torch.tensor(
    [total / (num_classes * label_counts[i]) for i in range(num_classes)],
    dtype=torch.float32
).to(device)

In [ ]:
def _get_head(model, arch_name):
    if arch_name == "mobilenet_v2":
        return model.classifier[-1]
    elif arch_name == "resnet50":
        return model.fc

def build_model(arch_name, num_classes):
    if arch_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    elif arch_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(f"Unknown architecture: {arch_name}")

    for param in model.parameters():
        param.requires_grad = False
    for param in _get_head(model, arch_name).parameters():
        param.requires_grad = True

    return model.to(device)

def set_backbone_trainable(model, trainable=True):
    for param in model.parameters():
        param.requires_grad = trainable

#### Third: TRAIN!

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    with torch.set_grad_enabled(is_train):
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1


def train_model(arch_name, num_classes, head_epochs=2, finetune_epochs=3, head_lr=1e-3, finetune_lr=1e-5):
    print(f"\n=== Training {arch_name} ===")
    model = build_model(arch_name, num_classes)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    history = []

    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=head_lr)
    for epoch in range(head_epochs):
        train_loss, train_acc, train_f1 = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_f1 = run_epoch(model, valid_loader, criterion)
        print(f"[head {epoch+1}/{head_epochs}] train_loss={train_loss:.3f} val_acc={val_acc:.3f} val_f1={val_f1:.3f}")
        history.append({"stage": "head", "epoch": epoch+1, "val_acc": val_acc, "val_f1": val_f1})

    set_backbone_trainable(model, True)
    optimizer = torch.optim.Adam(model.parameters(), lr=finetune_lr)
    for epoch in range(finetune_epochs):
        train_loss, train_acc, train_f1 = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_f1 = run_epoch(model, valid_loader, criterion)
        print(f"[finetune {epoch+1}/{finetune_epochs}] train_loss={train_loss:.3f} val_acc={val_acc:.3f} val_f1={val_f1:.3f}")
        history.append({"stage": "finetune", "epoch": epoch+1, "val_acc": val_acc, "val_f1": val_f1})

    return model, history

In [ ]:
dl_species_results = {}

for arch in ["mobilenet_v2", "resnet50"]:
    start = time.time()
    model, history = train_model(arch, num_classes=len(SPECIES_LIST))
    elapsed = time.time() - start

    dl_species_results[arch] = {
        "model": model, "history": history,
        "val_acc": history[-1]["val_acc"], "val_f1": history[-1]["val_f1"],
        "train_time_sec": elapsed,
    }
    print(f"{arch}: val_acc={history[-1]['val_acc']:.3f}, val_f1={history[-1]['val_f1']:.3f}, time={elapsed/60:.1f} min")

comparison_df_species = pd.DataFrame({
    arch: {"val_acc": r["val_acc"], "val_f1": r["val_f1"], "train_time_min": r["train_time_sec"]/60}
    for arch, r in dl_species_results.items()
}).T
print(comparison_df_species)

best_dl_species_name = comparison_df_species["val_f1"].idxmax()
best_dl_species_model = dl_species_results[best_dl_species_name]["model"]

#### Finally: Test on the best model

In [ ]:
best_dl_species_name = summary['best_model']

best_dl_species_model = build_model(best_dl_species_name, num_classes=len(SPECIES_LIST))
best_dl_species_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, f'best_dl_species_model_{best_dl_species_name}.pt')))
best_dl_species_model.eval()

In [ ]:
TEST_IMAGES_DIR = os.path.join(TEST_DIR, 'images')
test_dataset_dl = LeafFlatDataset(TEST_IMAGES_DIR, get_species_from_filename, SPECIES_TO_IDX, transform=eval_transform)
test_loader = DataLoader(test_dataset_dl, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

best_dl_species_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="test"):
        outputs = best_dl_species_model(imgs.to(device))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

pred_species = [IDX_TO_SPECIES[p] for p in all_preds]
true_species = [IDX_TO_SPECIES[l] for l in all_labels]

print(classification_report(true_species, pred_species))

cm = confusion_matrix(true_species, pred_species, labels=SPECIES_LIST)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=SPECIES_LIST, yticklabels=SPECIES_LIST)
plt.xticks(rotation=45, ha='right')
plt.title(f'{best_dl_species_name} - Species Test Confusion Matrix')
plt.show()

report = classification_report(true_species, pred_species, output_dict=True)
with open(os.path.join(OUTPUT_DIR, 'dl_species_summary.json'), 'w') as f:
    json.dump({"best_model": best_dl_species_name,
                "comparison": comparison_df_species.to_dict(orient="index"),
                "test_report": report}, f, indent=2)
torch.save(best_dl_species_model.state_dict(), os.path.join(OUTPUT_DIR, f'best_dl_species_model_{best_dl_species_name}.pt'))

In [ ]:
TEST_IMAGES_DIR = os.path.join(TEST_DIR, 'images')
test_dataset_dl = LeafFlatDataset(TEST_IMAGES_DIR, get_species_from_filename, SPECIES_TO_IDX, transform=eval_transform)
test_loader = DataLoader(test_dataset_dl, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

best_dl_species_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="test"):
        outputs = best_dl_species_model(imgs.to(device))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

pred_species = [IDX_TO_SPECIES[p] for p in all_preds]
true_species = [IDX_TO_SPECIES[l] for l in all_labels]

print(classification_report(true_species, pred_species))

cm = confusion_matrix(true_species, pred_species, labels=SPECIES_LIST)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=SPECIES_LIST, yticklabels=SPECIES_LIST)
plt.xticks(rotation=45, ha='right')
plt.title(f'{best_dl_species_name} - Species Test Confusion Matrix')
plt.show()

In [ ]:
import json
with open(os.path.join(OUTPUT_DIR, 'dl_species_summary.json')) as f:
    summary = json.load(f)

print(f"Best model: {summary['best_model']}\n")
print(pd.DataFrame(summary['comparison']).T)
print()
print(pd.DataFrame(summary['test_report']).T)